In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.show_dimensions", True)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

Tasks:
• Handle missing values (decide whether to drop, impute, or flag).

• Convert categorical fields to numeric (encoding).

• Normalize numerical features if needed.

• Create train/test split: use the most recent month of available data as the test set, 
and the X months immediately preceding it as the training set. X is not fixed —
treat the training window length as a tunable choice and experiment to determine 
the optimal value of X.

• Deliverable: 02_preprocessing.ipynb + cleaned CSV.

In [ ]:
"""
Load and combine CRMLS sold data from May 2025 to May 2026.
Restrict the dataset to California residential single-family properties.
"""

path = "../data/all_sales_data/"
target_months = [f"2025{m:02d}" for m in range(5, 13)] + [f"2026{m:02d}" for m in range(1, 6)]

all_files = []

for month in target_months:
    file_name = f"CRMLSSold{month}.csv"
    file_path = os.path.join(path, file_name)
    
    if os.path.exists(file_path):
        all_files.append(file_path)
    else:
        print(f"File not found: {file_path}")

if not all_files:
    raise FileNotFoundError("No files found for the specified months.")

df_raw = pd.concat(
    (
        pd.read_csv(
            f,
            low_memory=False,
            dtype={"PostalCode": "string"}
        )
        for f in all_files
    ),
    ignore_index=True
)

df_raw.columns = df_raw.columns.str.strip()

df_raw["ClosePrice"] = pd.to_numeric(df_raw["ClosePrice"], errors="coerce")
df_raw["CloseDate"] = pd.to_datetime(df_raw["CloseDate"], errors="coerce")

print("Raw shape:", df_raw.shape)

### Data Cleaning

In [ ]:
"""
Row filtering:
- Keep California properties only
- Keep Residential + SingleFamilyResidence only
- Remove missing or non-positive target values
"""

df = df_raw[
    (df_raw["StateOrProvince"] == "CA") &
    (df_raw["PropertyType"] == "Residential") &
    (df_raw["PropertySubType"] == "SingleFamilyResidence") &
    (df_raw["ClosePrice"].notna()) &
    (df_raw["ClosePrice"] > 0)
].copy()

print("After row filtering:", df.shape)

In [ ]:
"""
Observation:
"""
# LotSizeSquareFeet and LotSizeAcres are perfectly correlated, so only one should be kept.
# LotSizeArea is not strongly correlated with the other lot size fields and its unit is unclear.
# For the baseline model, I keep LotSizeSquareFeet and drop LotSizeAcres and LotSizeArea.

lot_cols = ["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]
lot_df = df[lot_cols].apply(pd.to_numeric, errors="coerce")

print("Missing rate:")
print(lot_df.isnull().mean())

print("\nCorrelation:")
display(lot_df.corr())

print("--------------")
#I removed Stories and kept Levels because the two variables are highly redundant, 
# but Levels has fewer missing values and captures more detailed structural information.
# While Stories mainly contains 1, 2, or missing values, Levels includes additional categories such as "ThreeOrMore" and "MultiSplit". 
# Therefore, Levels is more informative for the baseline model.
print(df[["Stories", "Levels"]].isnull().mean())
print(df["Stories"].value_counts(dropna=False))
print(df["Levels"].value_counts(dropna=False))
pd.crosstab(df["Stories"], df["Levels"], dropna=False)


In [ ]:
'''
Feature Selection
'''
#high missing_cols as columns with more than 60% missing values
missing_pct = df.isnull().mean()
high_missing_cols = missing_pct[missing_pct > 0.6]
print(high_missing_cols.sort_values(ascending=False))

#

drop_cols = set([
    # filter / status columns
    "PropertyType",
    "PropertySubType",
    "MlsStatus",
    "StateOrProvince", #CA only

    # identifiers
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "BuyerAgentMlsId",

    # agent / office / buyer fields
    "BuyerAgentAOR",
    "ListAgentAOR",
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "ListOfficeName",
    "BuyerOfficeName",
    "BuyerOfficeAOR",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "CoBuyerAgentFirstName",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",

    # leakage / listing-process columns
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",

    # exact address / noisy granular location
    "UnparsedAddress",
    "StreetNumberNumeric",

    #correlated columns
    "LotSizeAcres",
    "LotSizeArea",
    "Stories",

    #Coarse-grained regional data
    'CountyOrParish',
    'MLSAreaMajor', ## By observing, it does not contain much information.
    "City"
])

drop_cols.update(high_missing_cols.index.tolist())

# Keep the target variable
drop_cols.discard("ClosePrice")

df_current = df.drop(columns=list(drop_cols), errors="ignore")

print("After feature selection:", df_current.shape)
print(df_current.columns)


### Handling Invalid Numeric Values

In [ ]:
df["NewConstructionYN"].unique().tolist()

In [ ]:
numeric_cols = df_current.select_dtypes(include="number").columns.tolist()
df_num = df_current[numeric_cols]
display(df_num.describe().T)

In [ ]:
df_current0 = df_current.copy()

def apply_invalid_rule(df, col, condition, rule_name):
    before_missing = df[col].isnull().sum()
    
    df.loc[condition, col] = np.nan
    
    after_missing = df[col].isnull().sum()

    
    print(f"{col}: {after_missing - before_missing} additional missing values after {rule_name}.")
    
    return df

# Invalid coordinates for CA
df_current0 = apply_invalid_rule(df_current0, "Latitude", (df_current0["Latitude"] < 32) | (df_current0["Latitude"] > 42), "Invalid Latitude cleaning")
df_current0 = apply_invalid_rule(df_current0, "Longitude", (df_current0["Longitude"] < -125) | (df_current0["Longitude"] > -114), "Invalid Longitude cleaning")
# Invalid numeric property feature
df_current0 = apply_invalid_rule(df_current0, "LotSizeSquareFeet", (df_current0["LotSizeSquareFeet"] <= 0), "Invalid LotSizeSquareFeet cleaning")
df_current0 = apply_invalid_rule(df_current0, "LivingArea", (df_current0["LivingArea"] <= 0), "Invalid LivingArea cleaning")
df_current0 = apply_invalid_rule(df_current0, "ParkingTotal", (df_current0["ParkingTotal"] < 0), "Invalid ParkingTotal cleaning")
df_current0 = apply_invalid_rule(df_current0, "GarageSpaces", (df_current0["GarageSpaces"] < 0), "Invalid GarageSpaces cleaning")
df_current0 = apply_invalid_rule(df_current0, "BathroomsTotalInteger", (df_current0["BathroomsTotalInteger"] <= 0), "Invalid BathroomsTotalInteger cleaning")
df_current0 = apply_invalid_rule(df_current0, "BedroomsTotal", (df_current0["BedroomsTotal"] <= 0), "Invalid BedroomsTotal cleaning")
df_current0 = apply_invalid_rule(df_current0, "MainLevelBedrooms", (df_current0["MainLevelBedrooms"] < 0), "Invalid MainLevelBedrooms cleaning")
df_current0 = apply_invalid_rule(df_current0,"AssociationFee",df_current0["AssociationFee"] < 0,"invalid association fee cleaning")
df_current0 = apply_invalid_rule(df_current0,"MainLevelBedrooms",df_current0["MainLevelBedrooms"] > df_current0["BedroomsTotal"],"main-level bedrooms consistency cleaning")

display(df_current0.describe())


### Handling Count-Based Feature Sanity Checks

For count-based features, I used broad domain sanity checks instead of IQR-based thresholds. These variables are discrete counts and are usually concentrated around small integer values, so IQR can be too aggressive and may incorrectly flag valid larger homes as outliers.

`ParkingTotal` was given a higher upper bound because it may include driveway, open, or total available parking spaces. `GarageSpaces` specifically represents garage capacity, so it uses a stricter threshold.

Values above these broad upper bounds were converted to missing values rather than capped, because extremely large count values are more likely to be data quality issues than meaningful residential property characteristics.

This step is applied before the train/test split because the thresholds are manually defined domain rules, not thresholds learned from the data distribution.

In [ ]:
"""
Apply rule-based sanity checks to count-based features.

These rules are manually defined domain checks, so they can be applied before
the train/test split. Values that are clearly unrealistic are converted to NaN
instead of removing the full row.
"""

df_current = df_current0.copy()

def apply_count_upper_rule(df, col, upper_bound):
    """
    Convert unrealistic count-based feature values to NaN using broad
    domain sanity checks. These are feature values, so rows are not removed.
    """
    before_missing = df[col].isnull().sum()
    rows_above = (df[col] > upper_bound).sum()
    max_before = df[col].max()
    
    df.loc[df[col] > upper_bound, col] = np.nan
    
    after_missing = df[col].isnull().sum()
    
    print(f"{col}:")
    print(f"  upper bound: {upper_bound}")
    print(f"  max before cleaning: {max_before}")
    print(f"  rows converted to NaN: {rows_above}")
    print(f"  additional missing values: {after_missing - before_missing}")
    
    return df


count_upper_rules = {
    "ParkingTotal": 50,
    "GarageSpaces": 20,
    "BedroomsTotal": 20,
    "BathroomsTotalInteger": 20,
    "MainLevelBedrooms": 20
}

for col, upper_bound in count_upper_rules.items():
    if col in df_current.columns:
        df_current = apply_count_upper_rule(df_current, col, upper_bound)

print("Shape after count-based sanity checks:", df_current.shape)

### Cleaning Categorical Features

In this step, I performed light cleaning on the remaining categorical features before train/test splitting. Extra spaces were stripped, empty strings were converted to missing values, and `PostalCode` was standardized as a 5-digit categorical location feature.

`Flooring` was handled separately because it contains multi-value material combinations. Instead of one-hot encoding the raw flooring combinations, I converted it into material indicator features such as carpet, tile, wood, vinyl, laminate, stone, concrete, bamboo, and brick. I also created a separate indicator for `SeeRemarks`, because it is not a flooring material but may indicate that flooring information is available only in listing remarks.

For `HighSchoolDistrict`, non-specific labels such as `Other`, `See Remarks`, `Call Listing Office`, and multi-district inquiry notes were treated as missing values because they do not identify a specific school district.

Boolean-like fields such as `ViewYN`, `PoolPrivateYN`, `AttachedGarageYN`, `FireplaceYN`, and `NewConstructionYN` already contain only `True`, `False`, and missing values. I kept them as categorical variables rather than converting them directly to 0/1, because missing values may represent unreported information rather than a true false value.

Categorical imputation and one-hot encoding are not applied here. They will be fitted only on the training set after the train/test split to avoid data leakage.

In [ ]:
"""
Clean categorical features before train/test split.

Only rule-based string cleaning and row-level feature extraction are applied here.
Imputation and encoding will be handled after the train/test split.
"""

categorical_cols = df_current.select_dtypes(exclude="number").columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "CloseDate"]

# Strip spaces and convert empty strings to missing values
for col in categorical_cols:
    df_current[col] = df_current[col].astype("string").str.strip()
    df_current[col] = df_current[col].replace(r"^\s*$", np.nan, regex=True)


# PostalCode is a categorical location feature, not numeric.
# Keep only the first 5-digit ZIP code if ZIP+4 or mixed formats appear.
if "PostalCode" in df_current.columns:
    df_current["PostalCode"] = (
        df_current["PostalCode"]
        .astype("string")
        .str.extract(r"(\d{5})", expand=False)
    )


# Keep boolean-like categorical fields as True / False / missing.
# They are not converted to 0/1 because missing values may represent
# unreported information rather than a true False value.
yn_cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

for col in yn_cols:
    if col in df_current.columns:
        df_current[col] = df_current[col].astype("string").str.strip()

# Treat non-specific school district labels as missing.
# These values do not identify a specific district and are better handled
# as Unknown by the categorical imputer after the train/test split.
if "HighSchoolDistrict" in df_current.columns:
    ambiguous_school_pattern = (
        r"(?i)^(other|see remarks|call listing office|"
        r"more than 1 district.*|.*inquire.*)$"
    )
    
    df_current["HighSchoolDistrict"] = (
        df_current["HighSchoolDistrict"]
        .astype("string")
        .str.strip()
        .replace(r"^\s*$", np.nan, regex=True)
        .replace(ambiguous_school_pattern, np.nan, regex=True)
    )


# Convert Flooring from multi-value text into material indicator features.
# Use exact token matching because Flooring values are comma-separated material labels.
if "Flooring" in df_current.columns:
    flooring_clean = (
        df_current["Flooring"]
        .astype("string")
        .str.lower()
        .str.strip()
        .str.replace(r"\s*,\s*", ",", regex=True)
    )
    
    df_current["Flooring_missing"] = df_current["Flooring"].isna().astype(int)
    
    flooring_materials = {
        "has_carpet": "carpet",
        "has_tile": "tile",
        "has_wood": "wood",
        "has_vinyl": "vinyl",
        "has_laminate": "laminate",
        "has_stone": "stone",
        "has_concrete": "concrete",
        "has_bamboo": "bamboo",
        "has_brick": "brick"
    }
    
    for new_col, token in flooring_materials.items():
        pattern = rf"(?:^|,){re.escape(token)}(?:,|$)"
        df_current[new_col] = flooring_clean.str.contains(
            pattern,
            regex=True,
            na=False
        ).astype(int)
    
    # SeeRemarks is not a material, but it indicates that flooring details
    # may be described elsewhere in the listing.
    df_current["Flooring_see_remarks"] = flooring_clean.str.contains(
        r"(?:^|,)seeremarks(?:,|$)",
        regex=True,
        na=False
    ).astype(int)
    
    # Count how many actual flooring materials were identified.
    material_indicator_cols = list(flooring_materials.keys())
    df_current["Flooring_material_count"] = df_current[material_indicator_cols].sum(axis=1)
    
    # Drop the raw multi-value text column to avoid high-cardinality one-hot encoding.
    df_current = df_current.drop(columns=["Flooring"], errors="ignore")


print("Categorical cleaning completed.")
print("Shape after categorical cleaning:", df_current.shape)

In [ ]:
"""
Check categorical columns after basic categorical cleaning.
"""

categorical_cols = df_current.select_dtypes(exclude="number").columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "CloseDate"]

cat_summary = []

for col in categorical_cols:
    cat_summary.append({
        "column": col,
        "missing_count": df_current[col].isnull().sum(),
        "missing_rate": df_current[col].isnull().mean(),
        "unique_count": df_current[col].nunique(dropna=True),
        "top_value": df_current[col].mode(dropna=True).iloc[0] if df_current[col].notna().any() else np.nan,
        "top_value_count": df_current[col].value_counts(dropna=True).iloc[0] if df_current[col].notna().any() else 0
    })

cat_summary = pd.DataFrame(cat_summary).sort_values("unique_count", ascending=False)
display(cat_summary)

### Time-Based Train/Test Split

I used a time-based train/test split because the goal is to predict property close prices in a realistic future-data setting. The most recent available sale month is used as the test set, and all earlier months are used as the training set.

This split is placed before target outlier filtering, continuous feature capping, imputation, and encoding because those steps learn thresholds or mappings from the data and should be fitted only on the training set.

In [ ]:
"""
Create a time-based train/test split.

The most recent available CloseDate month is used as the test set.
All earlier months are used as the training set.
"""

df_model_base = df_current.copy()

df_model_base["CloseDate"] = pd.to_datetime(df_model_base["CloseDate"], errors="coerce")
df_model_base["SaleMonth"] = df_model_base["CloseDate"].dt.to_period("M")

test_month = df_model_base["SaleMonth"].max()

train_df = df_model_base[df_model_base["SaleMonth"] < test_month].copy()
test_df = df_model_base[df_model_base["SaleMonth"] == test_month].copy()

print("Test month:", test_month)
print("Train shape before target outlier handling:", train_df.shape)
print("Test shape:", test_df.shape)

### Handling Target Outliers in the Training Set

`ClosePrice` is the prediction target, so clearly abnormal target values should not be imputed. To avoid using information from the test set, target outlier thresholds are calculated using only the training data.

I created `PricePerLivingArea` only for target outlier inspection. This metric helps evaluate whether a sale price is reasonable relative to the property's living area. It is not used as a modeling feature because it is derived from the target variable and would cause target leakage.

The test set is left unchanged so that model evaluation remains closer to a realistic future-data setting.

In [ ]:
"""
Handle target outliers in the training set only.

The test set is not filtered here because it is used as the final evaluation set.
"""

train_df = train_df.copy()

# Create price per living area for target outlier inspection only
train_df["PricePerLivingArea"] = (
    train_df["ClosePrice"] / train_df["LivingArea"]
).replace([np.inf, -np.inf], np.nan)

print("Describe PricePerLivingArea for extreme low ClosePrice observations in training data:")
print(
    train_df.loc[
        train_df["ClosePrice"] < train_df["ClosePrice"].quantile(0.001),
        "PricePerLivingArea"
    ].describe()
)

print("---------")

print("Describe PricePerLivingArea for extreme high ClosePrice observations in training data:")
print(
    train_df.loc[
        train_df["ClosePrice"] > train_df["ClosePrice"].quantile(0.999),
        "PricePerLivingArea"
    ].describe()
)

# Low-end target outliers
low_price_threshold = train_df["ClosePrice"].quantile(0.001)
extreme_low_mask = train_df["ClosePrice"] < low_price_threshold

# High-end target outliers based on price per living area
high_price_threshold = train_df["ClosePrice"].quantile(0.999)
high_price_rows = train_df[train_df["ClosePrice"] > high_price_threshold].copy()

q1 = high_price_rows["PricePerLivingArea"].quantile(0.25)
q3 = high_price_rows["PricePerLivingArea"].quantile(0.75)
iqr = q3 - q1

if pd.notna(iqr) and iqr > 0:
    price_per_area_cutoff = q3 + 1.5 * iqr
else:
    price_per_area_cutoff = np.inf

extreme_high_mask = train_df["PricePerLivingArea"] > price_per_area_cutoff

print("Low ClosePrice threshold:", low_price_threshold)
print("Rows removed from train low end:", extreme_low_mask.sum())
print("Price per living area upper cutoff:", price_per_area_cutoff)
print("Rows removed from train high end:", extreme_high_mask.sum())

# Remove target outliers from training data only
train_df = train_df[
    (~extreme_low_mask) & (~extreme_high_mask)
].copy()

# Drop inspection-only variable to avoid target leakage
train_df = train_df.drop(columns=["PricePerLivingArea"], errors="ignore")

print("Train shape after target outlier handling:", train_df.shape)
print("Test shape unchanged:", test_df.shape)

### Splitting Features and Target

After the time-based split and target outlier handling on the training set, I separated the target variable from the feature matrix.

`ClosePrice` is used as the target. `CloseDate` and `SaleMonth` are removed from the feature matrix because they are used for splitting, not as baseline model predictors.

In [ ]:
"""
Separate features and target after train/test split.
"""

y_train = train_df["ClosePrice"]
y_test = test_df["ClosePrice"]

X_train = train_df.drop(columns=["ClosePrice", "CloseDate", "SaleMonth"], errors="ignore")
X_test = test_df.drop(columns=["ClosePrice", "CloseDate", "SaleMonth"], errors="ignore")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

### Train-Only Continuous Feature Capping

For highly skewed continuous features, I capped values above the 99.9th percentile. The cap thresholds are calculated using only the training set, then the same thresholds are applied to both the training and test sets.

This avoids data leakage because the test set is not used to determine preprocessing thresholds.

In [ ]:
"""
Fit continuous feature caps on X_train only, then apply the same caps to X_train and X_test.
"""

continuous_cap_cols = [
    "LivingArea",
    "LotSizeSquareFeet",
    "AssociationFee"
]

continuous_cap_cols = [col for col in continuous_cap_cols if col in X_train.columns]

cap_values = {}

for col in continuous_cap_cols:
    cap_values[col] = X_train[col].quantile(0.999)

def apply_continuous_caps(X, cap_values):
    """
    Apply pre-computed upper cap values to continuous features.
    The cap values should be fitted on the training set only.
    """
    X = X.copy()
    
    for col, cap in cap_values.items():
        if col in X.columns and pd.notna(cap):
            X[col] = X[col].clip(upper=cap)
    
    return X

X_train = apply_continuous_caps(X_train, cap_values)
X_test = apply_continuous_caps(X_test, cap_values)

cap_summary = pd.DataFrame({
    "column": list(cap_values.keys()),
    "train_q999_cap": list(cap_values.values()),
    "train_max_after_cap": [X_train[col].max() for col in cap_values.keys()],
    "test_max_after_cap": [X_test[col].max() for col in cap_values.keys()]
})

display(cap_summary)

### Train-Only Imputation and Encoding

Missing value imputation and categorical encoding are fitted only on the training set, then applied to both training and test sets.

Numeric features are imputed using the training median. Categorical features are imputed with `"Unknown"` and encoded using one-hot encoding. For high-cardinality categorical variables such as `PostalCode` and `HighSchoolDistrict`, rare categories are grouped through `min_frequency` to avoid creating too many sparse dummy variables.

The fitted preprocessing object stores the training-set medians, categorical mappings, and rare-category grouping rules, so the same transformations can be applied consistently to the test set and future prediction data.

In [ ]:
"""
Build preprocessing pipeline.

The preprocessor is fitted on X_train only and then used to transform both
X_train and X_test.

Before fitting the pipeline, pandas nullable missing values are converted to
regular np.nan values because scikit-learn imputers handle np.nan more reliably
than pandas pd.NA.
"""

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Make copies before preprocessing
X_train = X_train.copy()
X_test = X_test.copy()

# Convert pandas nullable string columns to object dtype
# This avoids pd.NA compatibility issues inside scikit-learn
string_cols = X_train.select_dtypes(include=["string"]).columns.tolist()

for col in string_cols:
    X_train[col] = X_train[col].astype(object)
    X_test[col] = X_test[col].astype(object)

# Convert all pandas missing values, including pd.NA, to np.nan
X_train = X_train.where(pd.notna(X_train), np.nan)
X_test = X_test.where(pd.notna(X_test), np.nan)

# Identify numeric and categorical columns after dtype cleanup
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

### Saving Preprocessing Artifacts

To make the preprocessing reusable, I saved the train-fitted preprocessing artifacts. `cap_values` stores the continuous feature cap thresholds learned from the training set. `preprocessor` stores the fitted numeric imputer, categorical imputer, one-hot encoder, and rare-category grouping rules.

These artifacts can be loaded later and applied directly to the test set or future prediction data without recalculating preprocessing rules from new data.

In [ ]:
"""
Save preprocessing artifacts for reuse in modeling and future prediction.
"""

import joblib

os.makedirs("../models", exist_ok=True)

preprocessing_artifacts = {
    "cap_values": cap_values,
    "preprocessor": preprocessor,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "feature_columns": X_train.columns.tolist()
}

joblib.dump(preprocessing_artifacts, "../models/preprocessing_artifacts.joblib")

print("Saved preprocessing artifacts to ../models/preprocessing_artifacts.joblib")